# aggregate — Rekap Hasil 5-Fold Student KD v3

Jalankan **setelah kelima fold selesai dievaluasi** (boleh juga sebagian — fold yang belum ada cuma jadi warning):

1. **Add Input**: kelima output notebook eval k-fold (yang berisi `student_kd_overall_fold{N}_9class.csv`). Tidak perlu dikumpulkan manual — CSV dicari otomatis rekursif di `/kaggle/input`.
2. **Run All** (CPU cukup, hitungan detik).
3. Hasil: `kfold_summary.csv` di panel Output + tabel markdown di bawah cell (mAP@0.5 & mAP@0.5:0.95 siang/malam + penurunan relatif `(mAP_siang − mAP_malam) / mAP_siang`, per fold + mean ± SD) — siap tempel langsung ke skripsi/paper.

In [1]:
# =============== KONFIGURASI ===============
# Folder yang di-scan untuk mencari student_kd_overall_fold{N}_9class.csv.
# Di Kaggle: add kelima output notebook eval sebagai Input, biarkan default.
SEARCH_DIRS = ["/kaggle/input", "/kaggle/working"]
OUT_PATH    = "/kaggle/working/kfold_summary.csv"
# ===========================================

import csv
import math
import os

K = 5
COLS = ["map50_day", "map50_night", "map5095_day", "map5095_night", "rel_drop"]
HEADERS = {"map50_day": "mAP@0.5 siang", "map50_night": "mAP@0.5 malam",
           "map5095_day": "mAP@0.5:0.95 siang", "map5095_night": "mAP@0.5:0.95 malam",
           "rel_drop": "penurunan relatif"}


def find_csv(fold):
    """Cari CSV overall fold N di mana pun dalam SEARCH_DIRS (rekursif)."""
    target = f"student_kd_overall_fold{fold}_9class.csv"
    for base in SEARCH_DIRS:
        if not os.path.isdir(base):
            continue
        for root, _dirs, files in os.walk(base):
            if target in files:
                return os.path.join(root, target)
    return None


def read_fold(path):
    rows = {r["split"]: r for r in csv.DictReader(open(path))}
    assert "daytime" in rows and "night" in rows, f"{path}: butuh baris daytime & night"
    d, n = rows["daytime"], rows["night"]
    m50d, m50n = float(d["map50"]), float(n["map50"])
    return {
        "map50_day": m50d, "map50_night": m50n,
        "map5095_day": float(d["map50_95"]), "map5095_night": float(n["map50_95"]),
        "rel_drop": (m50d - m50n) / m50d if m50d else float("nan"),
    }


def mean_sd(vals):
    m = sum(vals) / len(vals)
    sd = math.sqrt(sum((v - m) ** 2 for v in vals) / (len(vals) - 1)) if len(vals) > 1 else 0.0
    return m, sd


per_fold, missing = {}, []
for fi in range(K):
    p = find_csv(fi)
    if p:
        per_fold[fi] = read_fold(p)
        print(f"[INFO] fold_{fi}: {p}")
    else:
        missing.append(fi)
if missing:
    print(f"[WARNING] CSV fold {missing} tidak ditemukan — "
          f"rekap dihitung dari {len(per_fold)} fold yang tersedia.")
assert per_fold, "Tidak ada satu pun CSV fold yang ditemukan. Cek Input notebook."

stats = {c: mean_sd([per_fold[fi][c] for fi in sorted(per_fold)]) for c in COLS}

with open(OUT_PATH, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["fold"] + COLS)
    for fi in sorted(per_fold):
        w.writerow([f"fold_{fi}"] + [f"{per_fold[fi][c]:.4f}" for c in COLS])
    w.writerow(["mean"] + [f"{stats[c][0]:.4f}" for c in COLS])
    w.writerow(["sd"] + [f"{stats[c][1]:.4f}" for c in COLS])
print(f"[INFO] Tersimpan: {OUT_PATH}\n")

print("| fold | " + " | ".join(HEADERS[c] for c in COLS) + " |")
print("|---|" + "---:|" * len(COLS))
for fi in sorted(per_fold):
    print(f"| fold_{fi} | " + " | ".join(f"{per_fold[fi][c]:.4f}" for c in COLS) + " |")
print("| **mean ± SD** | " + " | ".join(
    f"**{stats[c][0]:.4f} ± {stats[c][1]:.4f}**" for c in COLS) + " |")


[INFO] fold_0: /kaggle/input/notebooks/keziameilanyt/kfold-0-eval/kd_output/student_kd_overall_fold0_9class.csv
[INFO] fold_1: /kaggle/input/notebooks/yosephoktavianus/eval-k-fold-student-kd-v3/kd_output/student_kd_overall_fold1_9class.csv
[INFO] fold_2: /kaggle/input/notebooks/stevieadrian/kd-2-eval/kd_output/student_kd_overall_fold2_9class.csv
[INFO] fold_3: /kaggle/input/notebooks/keziameilanyt/kfold-3-eval/kd_output/student_kd_overall_fold3_9class.csv
[INFO] fold_4: /kaggle/input/notebooks/keziameilany/k-fold-pt-3-eval/kd_output/student_kd_overall_fold4_9class.csv
[INFO] Tersimpan: /kaggle/working/kfold_summary.csv

| fold | mAP@0.5 siang | mAP@0.5 malam | mAP@0.5:0.95 siang | mAP@0.5:0.95 malam | penurunan relatif |
|---|---:|---:|---:|---:|---:|
| fold_0 | 0.5174 | 0.4814 | 0.2968 | 0.2612 | 0.0696 |
| fold_1 | 0.5249 | 0.4916 | 0.2968 | 0.2630 | 0.0634 |
| fold_2 | 0.5076 | 0.4800 | 0.2896 | 0.2644 | 0.0544 |
| fold_3 | 0.5153 | 0.4948 | 0.2927 | 0.2738 | 0.0398 |
| fold_4 | 0.5